# Turn AC-COLAB-BENCH-V1 -- Colab CUDA GPU benchmark runner (v2)

Embeds the pinned 5,000-sample benchmark set with `nlpai-lab/KURE-v1` (revision
`4ed4540949c70b7da2c74004a915e1f2d5e46e4f`, dimension 1024, float32, L2-normalized) and
writes a **binary** result package (`.npy` vectors + row-mapping/runtime/integrity
manifests -- never raw vectors in JSON), verified locally afterwards with
`node scripts/p11f0-colab-benchmark-verify.mjs`.

**What this notebook uploads/reads (non-sensitive only):**
- `embedding-input-manifest.summary.json` (git-tracked; `benchmark_sample.sample_ids` +
  pins only, no chunk text)
- `benchmark-sample-fulltext.jsonl` (task-owned; 5,000 rows of
  `{input_index, embedding_input_id, embed_text_sha256, text}` -- competition-corpus-derived
  embedding input text ONLY, never Gold/expected_answer/DEV_CHECK/HOLDOUT content)

**What it never does:** read/write Gold, DEV_CHECK, HOLDOUT, Owner decisions, API keys, DB
URLs, or the full raw DocumentIR/PostgreSQL dump. It embeds ONLY the 5,000 pinned texts, in
their fixed file order, and never logs prompt/text content -- only ids, hashes, counts, and
timings.

**This Turn does NOT execute this notebook or upload anything on the user's behalf** -- it is
a runnable template the user runs themselves in Colab, after which they download the four
output files and hand them to the local verifier.

In [ ]:
"""Cell 1 -- environment, GPU, and version fingerprint. Fails closed (raises,
does not continue) if no GPU is present or the model's actual loaded revision
does not match the pin. Logs versions/device only -- never text content."""
!pip install -q sentence-transformers==3.0.1

from google.colab import drive
drive.mount("/content/drive")

import json, time, hashlib, platform, struct
import numpy as np
import torch
import transformers
import sentence_transformers
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU required -- refusing to run on CPU"
GPU_NAME = torch.cuda.get_device_name(0)
CUDA_MEM_TOTAL_BYTES = torch.cuda.get_device_properties(0).total_memory

MODEL_REPOSITORY = "nlpai-lab/KURE-v1"
MODEL_REVISION = "4ed4540949c70b7da2c74004a915e1f2d5e46e4f"
DIMENSION = 1024
RUNNER = "COLAB_CUDA"
DRIVE_DIR = "/content/drive/MyDrive/p11f0-embedding-manifest"
BATCH_SIZE = 64  # bounded -- see cell 4 for the OOM-halves-and-restarts-the-whole-shard policy

model = SentenceTransformer(MODEL_REPOSITORY, revision=MODEL_REVISION, device="cuda")
loaded_dim = model.get_sentence_embedding_dimension()
if loaded_dim != DIMENSION:
    raise RuntimeError(f"REVISION_OR_DIMENSION_MISMATCH: loaded model reports dimension={loaded_dim}, expected {DIMENSION} -- refusing to proceed")

runtime_versions = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "sentence_transformers": sentence_transformers.__version__,
}
print(json.dumps({"gpu_name": GPU_NAME, "cuda_mem_total_bytes": CUDA_MEM_TOTAL_BYTES, "runtime_versions": runtime_versions, "model_revision_loaded": MODEL_REVISION}, indent=2))

In [ ]:
"""Cell 2 -- load and independently re-verify the pinned inputs: the sample
manifest's own sample_ids set, and every row's embed_text_sha256 against a
fresh sha256 of its own text (never trust the file's own claimed hash
without recomputing it)."""

def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

with open(f"{DRIVE_DIR}/embedding-input-manifest.summary.json") as f:
    summary = json.load(f)
expected_sample_ids = set(summary["benchmark_sample"]["sample_ids"])
if summary["model"]["repository"] != MODEL_REPOSITORY or summary["model"]["revision"] != MODEL_REVISION or summary["model"]["dimension"] != DIMENSION:
    raise RuntimeError("MANIFEST_PIN_MISMATCH: embedding-input-manifest.summary.json model pins do not match this notebook's MODEL_REPOSITORY/MODEL_REVISION/DIMENSION")

sample_rows = []
seen_ids = set()
with open(f"{DRIVE_DIR}/benchmark-sample-fulltext.jsonl") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        actual_sha = sha256_hex(row["text"])
        if actual_sha != row["embed_text_sha256"]:
            raise RuntimeError(f"TEXT_SHA_MISMATCH at input_index={row['input_index']}: file claims {row['embed_text_sha256']}, recomputed {actual_sha}")
        sample_rows.append(row)
        seen_ids.add(row["embedding_input_id"])

# Deterministic input ordering: sort explicitly by input_index so this cell
# never silently depends on the file being pre-sorted.
sample_rows.sort(key=lambda r: r["input_index"])

missing = expected_sample_ids - seen_ids
extra = seen_ids - expected_sample_ids
if missing or extra:
    raise RuntimeError(f"SAMPLE_SET_MISMATCH: missing={len(missing)} extra={len(extra)}")
if len(sample_rows) != len(expected_sample_ids):
    raise RuntimeError(f"ROW_COUNT_MISMATCH: expected {len(expected_sample_ids)}, got {len(sample_rows)}")

input_ordering_sha256 = sha256_hex("\n".join(r["embedding_input_id"] for r in sample_rows))
print(json.dumps({"row_count": len(sample_rows), "input_ordering_sha256": input_ordering_sha256}, indent=2))

In [ ]:
"""Cell 3 -- NPY writer (hand-implemented, matching
scripts/p11f0-colab-benchmark-verify.mjs's own reader exactly: float32,
C-order, '<f4' descr, v1.0 header)."""

def write_npy_float32_matrix(path, arr: np.ndarray):
    assert arr.dtype == np.float32 and arr.ndim == 2 and arr.flags["C_CONTIGUOUS"]
    header = f"{{'descr': '<f4', 'fortran_order': False, 'shape': ({arr.shape[0]}, {arr.shape[1]}), }}"
    pre_header_len = 6 + 2 + 2  # magic + version + 2-byte header-length field
    unpadded = header + "\n"
    total = pre_header_len + len(unpadded)
    padding = (64 - (total % 64)) % 64
    header_bytes = (header + " " * padding + "\n").encode("ascii")
    with open(path, "wb") as f:
        f.write(b"\x93NUMPY")
        f.write(bytes([1, 0]))
        f.write(struct.pack("<H", len(header_bytes)))
        f.write(header_bytes)
        f.write(arr.tobytes(order="C"))

def sha256_file(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

In [ ]:
"""Cell 4 -- the actual embedding run. Bounded batches, deterministic order,
atomic per-shard checkpoint (this notebook treats its whole 5,000-row input as
ONE shard -- the real 441,879-row full run uses the 8-way contiguous shard
plan, one notebook execution per shard, each independently atomic the same
way). On CUDA OOM: halve BATCH_SIZE and restart the CURRENT SHARD from
scratch -- never skip/drop the failing rows, never partially checkpoint a
shard. Vectors are L2-normalized by sentence-transformers' own
normalize_embeddings=True."""

def run_shard(rows, batch_size):
    texts = [r["text"] for r in rows]
    started = time.perf_counter()
    vectors = model.encode(
        texts, batch_size=batch_size, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=True,
    ).astype(np.float32)
    elapsed_ms = (time.perf_counter() - started) * 1000
    return vectors, elapsed_ms

batch_size = BATCH_SIZE
vectors = None
elapsed_ms = None
attempt = 0
while vectors is None:
    attempt += 1
    try:
        vectors, elapsed_ms = run_shard(sample_rows, batch_size)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if batch_size <= 1:
            raise RuntimeError("OOM_AT_BATCH_SIZE_1: cannot reduce further -- this shard cannot complete on this GPU")
        batch_size = max(1, batch_size // 2)
        print(f"[colab-cuda] CUDA OOM on attempt {attempt} -- halving batch_size to {batch_size} and restarting this shard from scratch (no partial/selective skip)")

non_finite = int((~np.isfinite(vectors)).sum())
if non_finite > 0:
    raise RuntimeError(f"NON_FINITE_VECTOR_COMPONENTS: {non_finite}")
norms = np.linalg.norm(vectors, axis=1)
worst_norm_dev = float(np.max(np.abs(norms - 1.0)))
if worst_norm_dev > 0.01:
    raise RuntimeError(f"NORMALIZATION_CHECK_FAILED: worst |L2 norm - 1| = {worst_norm_dev}")

print(json.dumps({"final_batch_size": batch_size, "attempts": attempt, "elapsed_ms": elapsed_ms, "worst_norm_deviation": worst_norm_dev}, indent=2))

In [ ]:
"""Cell 5 -- atomic write of the 4-file result package (vectors/mapping/
runtime/integrity), matching scripts/p11f0-colab-benchmark-verify.mjs's
expected filenames and schema exactly."""

prefix = "colab-cuda"
vector_path = f"{DRIVE_DIR}/{prefix}-vectors.npy"
mapping_path = f"{DRIVE_DIR}/{prefix}-row-mapping.jsonl"
runtime_path = f"{DRIVE_DIR}/{prefix}-runtime-manifest.json"
integrity_path = f"{DRIVE_DIR}/{prefix}-file-integrity-manifest.json"

write_npy_float32_matrix(vector_path, vectors)

with open(mapping_path, "w") as f:
    for i, row in enumerate(sample_rows):
        f.write(json.dumps({"input_index": i, "embedding_input_id": row["embedding_input_id"], "embed_text_sha256": row["embed_text_sha256"]}) + "\n")

runtime_manifest = {
    "schema_version": "p11f0-colab-benchmark-local-reference-run.v1",
    "model": {"repository": MODEL_REPOSITORY, "revision": MODEL_REVISION, "dimension": DIMENSION},
    "device": GPU_NAME,
    "runtime_versions": runtime_versions,
    "batch_size": batch_size,
    "row_count_requested": len(sample_rows),
    "row_count_succeeded": len(sample_rows),
    "row_count_failed": 0,
    "elapsed_ms_total": elapsed_ms,
    "texts_per_sec": len(sample_rows) / (elapsed_ms / 1000),
    "cuda_mem_total_bytes": CUDA_MEM_TOTAL_BYTES,
    "input_ordering_sha256": input_ordering_sha256,
    "vector_output_sha256": sha256_file(vector_path),
    "normalization": "l2 (unit-norm, sentence-transformers normalize_embeddings=True)",
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(runtime_path, "w") as f:
    json.dump(runtime_manifest, f, indent=2)

integrity_manifest = {
    "schema_version": "p11f0-colab-benchmark-file-integrity.v1",
    "files": {
        f"{prefix}-vectors.npy": sha256_file(vector_path),
        f"{prefix}-row-mapping.jsonl": sha256_file(mapping_path),
        f"{prefix}-runtime-manifest.json": sha256_file(runtime_path),
    },
}
with open(integrity_path, "w") as f:
    json.dump(integrity_manifest, f, indent=2)

print(json.dumps({"wrote": [vector_path, mapping_path, runtime_path, integrity_path], "runtime_manifest": runtime_manifest}, indent=2))
print("\nNext step (run LOCALLY, not in Colab): download these 4 files, then:\n"
      "  node scripts/p11f0-colab-benchmark-verify.mjs <summary.json> <sample-fulltext.jsonl> <local-dir> <this-download-dir> colab-cuda")